<a href="https://colab.research.google.com/github/malyadri6761/RAG/blob/master/rag_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

**FAISS**

---

```
  FAISS IndexFlatL2 uses Euclidean (L2) distance.
  Query embedding
      │
      ├──── close ────> relevant chunk
      │
      ├───────────────> somewhat relevant
      │
      └────────────────────> less relevant
  
  For L2 distance => Smaller distance = more similar


```

In [3]:
!pip install sentence-transformers faiss-cpu groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 8.7 MB/s eta 0:00:00


In [4]:
from sentence_transformers import SentenceTransformer
import faiss

from groq import Groq
from google.colab import userdata

In [6]:
groq_api_key = userdata.get('GROQ_API_KEY')
model = Groq(api_key = groq_api_key)

In [7]:
# ============================================================
# 4. CREATE SAMPLE DOCUMENTS
# ============================================================

docs = [

    {
        "text": """
        Artificial Intelligence is a field of computer science that focuses
        on creating systems capable of performing tasks that normally require
        human intelligence.

        Machine learning is a subset of artificial intelligence. It allows
        computers to learn patterns from data and make predictions or decisions
        without being explicitly programmed for every task.

        Deep learning is a subset of machine learning that uses neural networks
        with multiple layers to learn complex patterns from large amounts of data.
        """,

        "source": "ai.txt"
    },

    {
        "text": """
        Supervised learning is a machine learning approach where a model learns
        from labeled training data.

        Classification is a supervised learning task where the goal is to predict
        a discrete class or category.

        Regression is a supervised learning task where the goal is to predict
        a continuous numerical value.

        Unsupervised learning works with data that does not have labeled outputs.

        Clustering is one example of unsupervised learning.
        """,

        "source": "ml.txt"
    }
]

In [8]:
def create_chunks(text,chunk_size=10):
  words = text.split()
  chunks = []

  for i in range(0,len(words),chunk_size):
    chunk = " ".join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks

In [9]:
all_chunks = []

for document in docs:
  chunks = create_chunks(document["text"])
  for chunk in chunks:
    all_chunks.append({
        "text" : chunk,
        "source" : document["source"]
    })


In [10]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(texts)
print(embeddings.shape)

(15, 384)


In [21]:
texts

['Artificial Intelligence is a field of computer science that focuses',
 'on creating systems capable of performing tasks that normally require',
 'human intelligence. Machine learning is a subset of artificial intelligence.',
 'It allows computers to learn patterns from data and make',
 'predictions or decisions without being explicitly programmed for every task.',
 'Deep learning is a subset of machine learning that uses',
 'neural networks with multiple layers to learn complex patterns from',
 'large amounts of data.',
 'Supervised learning is a machine learning approach where a model',
 'learns from labeled training data. Classification is a supervised learning',
 'task where the goal is to predict a discrete class',
 'or category. Regression is a supervised learning task where the',
 'goal is to predict a continuous numerical value. Unsupervised learning',
 'works with data that does not have labeled outputs. Clustering',
 'is one example of unsupervised learning.']

In [12]:
# FAISS INDEX
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)
index.add(embeddings)
print(index.ntotal)

15


In [13]:
def retrieve(query,k=3):
  query_embedding = embedding_model.encode([query])

  distance,indices = index.search(
      query_embedding,
      k
  )

  results = []
  for rank,idx in enumerate(indices[0]):
    results.append({
        "rank" : rank,
        "source" : all_chunks[idx]["source"],
        "text" : all_chunks[idx]["text"],
        "distance" : distance[0][rank]
    })
  return results

In [14]:
query = "What is supervised learning?"

results = retrieve(
    query,
    k=3
)

In [26]:
distance,indeices = index.search(embedding_model.encode([query]),3)
for idx in indeices[0]:
  print(all_chunks[idx])


{'text': 'Supervised learning is a machine learning approach where a model', 'source': 'ml.txt'}
{'text': 'learns from labeled training data. Classification is a supervised learning', 'source': 'ml.txt'}
{'text': 'is one example of unsupervised learning.', 'source': 'ml.txt'}


In [15]:
# ============================================================
# 12. DISPLAY RETRIEVED CHUNKS
# ============================================================

for result in results:

    print("Rank:", result["rank"])
    print("Distance:", result["distance"])
    print("Source:", result["source"])
    print("Text:", result["text"])

    print("-" * 60)

Rank: 0
Distance: 0.29673254
Source: ml.txt
Text: Supervised learning is a machine learning approach where a model
------------------------------------------------------------
Rank: 1
Distance: 0.41850287
Source: ml.txt
Text: learns from labeled training data. Classification is a supervised learning
------------------------------------------------------------
Rank: 2
Distance: 0.9468646
Source: ml.txt
Text: is one example of unsupervised learning.
------------------------------------------------------------


In [16]:
# ============================================================
# 13. CREATE CONTEXT
# ============================================================

context = "\n\n".join(
    result["text"]
    for result in results
)

print(context)

Supervised learning is a machine learning approach where a model

learns from labeled training data. Classification is a supervised learning

is one example of unsupervised learning.


In [17]:
# ============================================================
# 14. CREATE RAG PROMPT
# ============================================================

prompt = f"""
You are a helpful AI assistant.

Answer the user's question using only the provided context.

If the answer is not present in the context,
say:

"I don't know based on the provided documents."

Do not use outside knowledge.

Context:
{context}

Question:
{query}

Answer:
"""

In [19]:
# ============================================================
# 15. GROQ GENERATION
# ============================================================

response = model.chat.completions.create(

    model="openai/gpt-oss-120b",

    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],

    temperature=0
)

In [20]:
# ============================================================
# 16. FINAL ANSWER
# ============================================================

answer = response.choices[0].message.content

print("Answer:")
print(answer)

Answer:
Supervised learning is a machine learning approach where a model learns from labeled training data.
